[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/22_capstone/22_proyecto_capstone.ipynb)

# Proyecto 22 · Entrega final: la plataforma de SupportOps y su revisión de arquitectura

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 2 h |
| **Servicios de Compose** | Ninguno. Docker opcional para validar tu Collector con `otelcol validate` (se marca si no hay) |
| **RAM aproximada** | < 700 MB |
| **Prerrequisitos** | Lección `22_capstone` y todo el curso |
| **Conexión** | Integra los proyectos 03-21: lo que construiste por separado, ahora **en una sola plataforma**, medida con su propia telemetría y defendida por escrito |

## Contexto de negocio

SupportOps pasa a producción con el sistema multi-agente. Dirección pide: (1) un **gateway** del Collector que integre redacción, métricas previas al muestreo, tail sampling y exportación; (2) **contratos de telemetría** que corran en CI; (3) un informe de **SLO y calidad** de la ejecución con incidente;
(4) un **scorecard** y un checklist de producción honestos; (5) una **revisión de arquitectura** que otro ingeniero pueda atacar.

## Datos

El **Paso 0** ejecuta la plataforma completa (multi-agente, `mock`, semilla 42, 480 tickets en 30 min, caída parcial de `get_order` del `billing-agent` entre los minutos 10 y 20) y te da `MESH`, `TICKETS`, `RES`, el corpus de muestreo `C` y los resultados de referencia. Todo en memoria.

## Entregables y criterios de evaluación (rúbrica integral)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución; `mock`, semilla 42) |
|---|---|---|---|
| R1 | `gateway_config` | YAML válido; orden correcto de procesadores en los 4 pipelines; span_metrics antes de tail_sampling; `ottl_condition` (no `numeric_attribute`) en la política de coste; `otelcol validate` (si Docker) | 6/6 |
| R2 | `telemetry_contracts` | Detecta **bien** la plataforma sana (todos OK) y **detecta** una plataforma con propagación rota y otra con PII sin redactar | 5/5 |
| R3 | `slo_report` | Disponibilidad, p95 y SLI de latencia exactos; alerta multiventana (5 min/1 min, 3×): primer disparo dentro de la caída (min 10-22) y sin falsas alarmas | 5/5 |
| R4 | `quality_report` | Estimación por muestra uniforme con IC de Wilson: contiene a la verdad y cumple el objetivo | 3/3 |
| R5 | `sampling_report` | Recall de errores = 100 %, fracción de trazas guardadas ≤ 35 %, y tasa de error corregida a ±15 % de la verdad | 3/3 |
| R6 | `cost_report` | Coste por agente solo en hojas = coste del árbol (tolerancia relativa 1e-3: cada resultado redondea a 6 decimales) y el doble conteo se detecta | 3/3 |
| R7 | `checklist_status` | 31 puntos clasificados; los automáticos coinciden con el scorecard; ≥ 12 «evidencia entregada» y 3 «pendiente» | 4/4 |
| R8 | `architecture_review` | Documento con las 8 secciones, ≥ 6 riesgos con señal y mitigación, coste con sensibilidad (×2) y plan de pruebas con la mesa | 6/6 |

> 📏 Umbrales calibrados con la solución de referencia en `mock` con la semilla 42 fija; el resto de valores exactos se imprimen en el detalle de la rúbrica.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import collections, dataclasses, hashlib, json, math, re, statistics, subprocess, tempfile, time, uuid, pathlib
import yaml
from IPython.display import Markdown
from supportops import incidents as I, privacy as P, sampling as S
from supportops.multiagent import build_mesh, MultiAgentSupportOps, DelegationPolicy, cost_rollup, waterfall_services
from supportops.traffic import Incident
from supportops.pricing import cost_usd
from supportops.quality_eval import wilson_interval
STACK = check_stack([])
LIVE = False
DOCKER = P.docker_available()
PLATFORM = pathlib.Path(ROOT) / "22_capstone" / "platform"

In [ ]:
def run_platform(n: int = 480, seed: int = SEED, redact: bool = True, content: bool = True, mode: str = "memory", endpoint: str | None = None):
    """La plataforma completa de SupportOps: multi-agente (orquestador + 4 especialistas) con telemetría OTel por servicio, captura de contenido configurable,
    redacción en la app, políticas de resiliencia y una caída parcial (la herramienta `get_order` del billing-agent agota plazos entre los minutos 10 y 20 de una ejecución de 30 min). Devuelve (mesh, tickets, resultados)."""
    mesh = build_mesh(mode=mode, endpoint=endpoint)
    red = P.Redactor("hash", names=True) if redact else None
    for a in mesh.handles:
        mesh.instr(a, capture_content=content, redact=red)
    system = MultiAgentSupportOps(mesh, mode="delegate", policy=DelegationPolicy(timeout_s=6, max_retries=2, backoff_s=0.5))
    inc = Incident("billing-timeouts", "tool_timeout", 10 * 60, 20 * 60, 1.0, {"tool": "get_order", "timeout_s": 8.0, "agent": "billing-agent", "sticky": True})
    tickets = generate_tickets(n, seed, duration_s=1800, fault_rates=DEFAULT_FAULT_RATES, incidents=[inc], pii_rate=0.3)
    results = [system.run(t) for t in tickets]
    mesh.force_flush()
    return mesh, tickets, results

In [ ]:
def pii_after(mesh):
    txt = [json.dumps(s["attributes"], ensure_ascii=False) for s in mesh.span_records()] + [str(l["body"]) + json.dumps(l["attributes"], ensure_ascii=False) for l in mesh.log_records()]
    return P.pii_exposure(txt)

def scorecard(mesh, tickets, results, corpus_cov=None, cfg_ok=None, quality=None):
    """Comprobaciones automáticas de la plataforma con SU PROPIA telemetría. Devuelve una lista de filas (id del checklist, área, medida, umbral, ok)."""
    sp, lg = mesh.span_records(), mesh.log_records()
    rows = []
    def add(cid, area, what, value, thr, ok): rows.append({"id": cid, "área": area, "comprobación": what, "valor": value, "umbral": thr, "ok": bool(ok)})
    tt = I.trace_table(sp, tickets[0].created_at - 1)
    add("I1", "Instrumentación", "trazas por ticket", round(len(tt) / len(tickets), 3), "= 1", len(tt) == len(tickets))
    leaf = collections.defaultdict(int)
    for s in sp:
        if s["name"].startswith("chat"): leaf[s["trace_id"]] += int(s["attributes"].get("gen_ai.usage.input_tokens", 0) or 0)
    tree = sum(r.input_tokens for r in results)
    add("I3", "Instrumentación", "tokens de entrada: hojas chat / árbol del orquestador", round(sum(leaf.values()) / tree, 3), "= 1", sum(leaf.values()) == tree)
    cov = np.mean([bool(l["trace_id"]) for l in lg]); add("I6", "Instrumentación", "logs con trace_id", round(float(cov), 3), "= 1", cov == 1.0)
    good = np.mean([r.status in ("ok", "escalated") for r in results]); add("S1", "SLOs", "disponibilidad (objetivo 99 %: se mide, no se exige cumplirlo en el incidente)", round(float(good), 4), "medida", True)
    p95 = float(np.percentile([r.latency_s for r in results], 95)); add("S1", "SLOs", "p95 de la petición (s)", round(p95, 2), "medida", True)
    c = [r.cost_usd for r in results]; add("K1", "Coste", "coste por ticket (USD, ilustrativo) y p95/media", f"{np.mean(c):.5f} · {np.percentile(c, 95) / np.mean(c):.2f}×", "medida", True)
    ex = pii_after(mesh); no_names = sum(v for k, v in ex.items() if k != "name")
    add("P2", "Privacidad", "PII residual en spans+logs (sin nombres)", no_names, "= 0", no_names == 0)
    if quality is not None: add("Q3", "Calidad", "groundedness (muestra uniforme 25 %): límite inferior del IC 95 %", round(quality["lo"], 3), "≥ 0,85", quality["lo"] >= 0.85)
    if corpus_cov is not None:
        add("C3", "Collector", "recall de errores tras el tail sampling (%)", round(corpus_cov["recall_error_pct"], 1), "= 100", corpus_cov["recall_error_pct"] >= 99.9)
    if cfg_ok is not None: add("C2", "Collector", "configuración del gateway válida (otelcol validate)", cfg_ok, "OK", cfg_ok is True)
    return pd.DataFrame(rows)

## Paso 0 · La plataforma en marcha (te la damos hecha)

In [ ]:
N = 480
MESH, TICKETS, RES = run_platform(N)
SP, LG = MESH.span_records(), MESH.log_records()
C = S.build_corpus(1200, SEED, incidents=[Incident("provider", "llm_error", 300, 420, 0.6, {"status": 503, "at_call": 0, "persistent": True})])
T0 = 1_788_253_200.0
print(f"{len(TICKETS)} tickets · {len(SP)} spans · {len(LG)} logs · caída parcial: minutos 10-20 · corpus de muestreo: {len(C)} trazas")

## Paso 1 · El gateway del Collector

**TODO 1.** `gateway_config(blocked_values, policies) -> str` devuelve el YAML (con `yaml.safe_dump`) de un Collector con:
receptor `otlp` (gRPC 4317 y HTTP 4318); procesadores `memory_limiter`, `redaction` (`allow_all_keys: true`, `blocked_values`, `summary: info`), `tail_sampling` (con `policies`) y `batch`; conectores `forward` y `span_metrics`;
exportadores `otlp_grpc/tempo` (`tempo:4317`, `tls.insecure: true`), `otlp_http/loki` (`http://loki:3100/otlp`) y `otlp_http/prometheus` (`http://prometheus:9090/api/v1/otlp`); y los pipelines `traces/in` (otlp → `memory_limiter`, `redaction` → `forward` y `span_metrics`),
`traces/sampled` (forward → `tail_sampling`, `batch` → tempo), `metrics` (otlp y span_metrics → `memory_limiter`, `batch` → prometheus) y `logs` (otlp → `memory_limiter`, `redaction`, `batch` → loki).

<details>
<summary>🪜 Pista 1</summary>

Estructura: `receivers`, `processors`, `connectors`, `exporters`, `service.pipelines`. El orden de la lista `processors` de cada pipeline es el orden de ejecución.

</details>

<details>
<summary>🪜 Pista 2</summary>

Las políticas se pasan ya hechas (`S.tail_policy_config(policies)` de `supportops.sampling` te da la lista de dicts: úsala tal cual). Para validar: `P.validate_collector_config(cfg)` (Docker).

</details>

In [ ]:
POLICIES = [S.StatusCode(("ERROR",), name="errors"), S.Latency(5000, name="slow"), S.AttrCompare(key="supportops.tree.cost_usd", op=">", value=0.004, name="costly"), S.Probabilistic(10, name="baseline")]
BLOCKED = P.go_blocked_values()

def gateway_config(blocked_values, policies) -> str:
    # TODO 1
    raise NotImplementedError("TODO 1")

try:
    CFG = gateway_config(BLOCKED, S.tail_policy_config(POLICIES)); print(CFG[:500])
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 2 · Contratos de telemetría

**TODO 2.** `telemetry_contracts(spans, logs, tickets, results) -> dict[str, bool]` devuelve estas cinco comprobaciones:

- `una_traza_por_ticket`: nº de `trace_id` distintos de los spans == nº de tickets;
- `hojas_igual_arbol`: suma de `gen_ai.usage.input_tokens` de los spans `chat` == suma de `input_tokens` de los resultados;
- `logs_con_trace_id`: todos los logs tienen `trace_id` no vacío;
- `sin_pii_sin_nombres`: ninguna PII (con `P.pii_exposure`, ignorando `name`) en los atributos de los spans ni en el cuerpo y atributos de los logs;
- `workflow_raiz`: toda traza tiene exactamente una raíz y se llama `invoke_workflow …`.

In [ ]:
def telemetry_contracts(spans, logs, tickets, results) -> dict:
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    print(telemetry_contracts(SP, LG, TICKETS, RES))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · SLO y alertas

**TODO 3.** `slo_report(results, t0) -> dict` devuelve `{"availability", "p95_s", "latency_sli", "first_fire", "false_alarms"}`: disponibilidad = fracción de `status` ∈ {`ok`, `escalated`}; `p95_s` = `np.percentile(latency_s, 95)`; `latency_sli` = fracción de `latency_s ≤ 10`.
La alerta es sobre el **SLI de latencia** (objetivo 95 %): por minuto (`int((r.start_s − t0)//60)`, 30 minutos) cuenta buenos y totales y calcula el burn rate con ventanas de **5 min** (larga) y **1 min** (corta), solo con ≥ 5 eventos en la ventana; dispara si **ambos** ≥ 3,0 (política de ticket: una caída parcial no quema a 14,4×).
`first_fire` = primer minuto con alerta (o `None`) y `false_alarms` = nº de minutos con alerta **antes del minuto 10**.

<details>
<summary>🪜 Pista 1</summary>

`burn = (1 − buenos/totales)/(1 − objective)` sobre la suma de los últimos `w` minutos (incluido el actual).

</details>

In [ ]:
def slo_report(results, t0) -> dict:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    print(slo_report(RES, T0))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Calidad con intervalo

**TODO 4.** `quality_report(results, every=4) -> dict` toma la **muestra uniforme determinista** de los resultados con `int(sha256(ticket_id).hexdigest(), 16) % every == 0`, y devuelve `{"n", "p", "lo", "hi", "truth", "covers", "meets_target"}`:
`p = k/n` con `k` = `grounded` verdaderos; `lo, hi` = intervalo de Wilson al 95 % (`wilson_interval(k, n)` de `supportops.quality_eval`); `truth` = fracción de `grounded` en **toda** la población; `covers` = `lo ≤ truth ≤ hi`; `meets_target` = `lo ≥ 0,85`.

In [ ]:
def quality_report(results, every=4) -> dict:
    # TODO 4
    raise NotImplementedError("TODO 4")

try:
    print(quality_report(RES))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 5 · Muestreo

**TODO 5.** `sampling_report(corpus, policies) -> dict` ejecuta `S.TailSampler(policies, decision_wait_s=10, num_traces=50_000)` sobre `corpus.traces`, y devuelve `{"recall_error", "kept_pct", "error_rate_truth", "error_rate_naive", "error_rate_weighted"}`: `recall_error` y `kept_pct` de `S.coverage(kept, corpus)`
(`recall_error_pct`, `kept_traces_pct`); las tasas con `S.estimate_metrics`: verdad (todas las trazas), ingenua (solo las guardadas) y ponderada con `S.inclusion_weights(kept, lambda t: t.interesting, 0.10)`.

In [ ]:
def sampling_report(corpus, policies) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    print(sampling_report(C, POLICIES))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · Coste

**TODO 6.** `cost_report(spans) -> dict` devuelve `{"by_service", "total_leaf", "total_naive"}`: `by_service` con `supportops.multiagent.cost_rollup(spans)` (solo hojas `chat`); `total_leaf` = suma de sus `cost_usd`;
`total_naive` = el coste que saldría de sumar **todos** los spans con `gen_ai.usage.input_tokens/output_tokens` (usa `cost_usd(modelo, in, out)` con `gen_ai.response.model` o `gen_ai.request.model`, y `"mock-small"` si faltan).

In [ ]:
def cost_report(spans) -> dict:
    # TODO 6
    raise NotImplementedError("TODO 6")

try:
    r_ = cost_report(SP); print(round(r_["total_leaf"], 4), round(r_["total_naive"], 4))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 7 · Checklist de producción

**TODO 7.** `checklist_status(items, measured_ok, measured_all, evidence_ids) -> list[dict]` clasifica cada ítem de `platform/production-checklist.yaml`: si `automatica`: `"medido"` si su `id` está en `measured_ok`, `"falla"` si está en `measured_all` pero no en `measured_ok`, `"sin medir"` si no está en `measured_all`;
si no es automática: `"evidencia"` si su `id` está en `evidence_ids` y `"pendiente"` en otro caso. Devuelve los ítems con una clave `estado` añadida.

In [ ]:
CHECKLIST = yaml.safe_load((PLATFORM / "production-checklist.yaml").read_text())["items"]
EVID = {"I2", "I4", "I5", "I7", "C4", "C5", "S3", "Q1", "Q2", "K2", "P3", "P4", "D2", "D3"}      # artefactos entregados por el curso (tuyos al terminar los módulos)

def checklist_status(items, measured_ok, measured_all, evidence_ids) -> list:
    # TODO 7
    raise NotImplementedError("TODO 7")

try:
    sc_ = scorecard(MESH, TICKETS, RES); print(collections.Counter(x["estado"] for x in checklist_status(CHECKLIST, set(sc_[sc_.ok]["id"]), set(sc_["id"]), EVID)))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 8 · Revisión de arquitectura

**TODO 8.** `architecture_review(facts) -> str` genera el documento **Markdown** de la revisión a partir de los hechos medidos (`facts`: dict con `slo`, `quality`, `sampling`, `cost`, `contracts`, `checklist`). Debe contener, como encabezados `## `, las secciones de la plantilla
(`platform/architecture-review-template.md`: Contexto y objetivos, Requisitos y restricciones, Arquitectura propuesta, Decisiones clave, Riesgos y modos de fallo, Coste, Plan de despliegue y de pruebas, Preguntas abiertas), una **tabla de riesgos con ≥ 6 filas** (riesgo | señal que lo detecta | mitigación),
una sección de **coste con sensibilidad** (menciona `×2` de tráfico) con cifras de `facts["cost"]`, y un plan de pruebas que mencione la **mesa de incidentes**. Usa cifras reales de `facts`, no inventadas.

In [ ]:
def architecture_review(facts: dict) -> str:
    # TODO 8
    raise NotImplementedError("TODO 8")

try:
    print(architecture_review({"slo": {}, "quality": {}, "sampling": {}, "cost": {}, "contracts": {}, "checklist": {}})[:300])
except NotImplementedError as e:
    print("Pendiente:", e)

---
## Rúbrica automática

Ejecuta la celda cuando hayas terminado los TODO.

In [ ]:
def run_rubric():
    rows = []
    def add(rid, name, ok, detail=""):
        rows.append({"id": rid, "criterio": name, "resultado": "✅" if ok else "❌", "detalle": detail})
    # R1
    try:
        cfg = gateway_config(BLOCKED, S.tail_policy_config(POLICIES)); y = yaml.safe_load(cfg); pl = y["service"]["pipelines"]
        pol = y["processors"]["tail_sampling"]["policies"]
        checks = {"4 pipelines": set(pl) >= {"traces/in", "traces/sampled", "metrics", "logs"},
                  "memory_limiter primero": all(pl[k]["processors"][0] == "memory_limiter" for k in ("traces/in", "metrics", "logs")),
                  "redaction en traces/in y logs, antes de batch": pl["traces/in"]["processors"] == ["memory_limiter", "redaction"] and pl["logs"]["processors"][:2] == ["memory_limiter", "redaction"],
                  "span_metrics antes del muestreo": "span_metrics" in pl["traces/in"]["exporters"] and "forward" in pl["traces/in"]["exporters"] and pl["traces/sampled"]["processors"][0] == "tail_sampling",
                  "ottl_condition y sin numeric_attribute": any(p.get("type") == "ottl_condition" for p in pol) and not any(p.get("type") == "numeric_attribute" for p in pol)}
        if DOCKER:
            ok_v, msg = P.validate_collector_config(cfg); checks["otelcol validate"] = ok_v
        else:
            checks["otelcol validate (omitido sin Docker)"] = True
        add("R1", "gateway_config: 6 comprobaciones", all(checks.values()), str({a: b for a, b in checks.items() if not b} or "todas OK"))
    except Exception as e: add("R1", "gateway_config", False, f"{type(e).__name__}: {e}")
    # R2
    try:
        good = telemetry_contracts(SP, LG, TICKETS, RES)
        broken_mesh, bt, br = run_platform(40, redact=True)
        # plataforma con propagación rota: ejecutamos con propagation=none
        mesh_b = build_mesh(mode="memory"); sysb = MultiAgentSupportOps(mesh_b, mode="delegate", propagation="none"); tb = generate_tickets(30, SEED, duration_s=300, pii_rate=0.0); rb = [sysb.run(t) for t in tb]; mesh_b.force_flush()
        broken = telemetry_contracts(mesh_b.span_records(), mesh_b.log_records(), tb, rb)
        raw_mesh, rt, rr = run_platform(40, redact=False, content=True)
        leaky = telemetry_contracts(raw_mesh.span_records(), raw_mesh.log_records(), rt, rr)
        checks = {"sana: 5/5": all(good.values()) and len(good) == 5, "propagación rota detectada": not broken["una_traza_por_ticket"] and not broken["workflow_raiz"], "PII sin redactar detectada": not leaky["sin_pii_sin_nombres"],
                  "la plataforma rota sigue cumpliendo hojas=árbol": broken["hojas_igual_arbol"], "la que filtra PII sigue con una traza por ticket": leaky["una_traza_por_ticket"]}
        add("R2", "telemetry_contracts: 5 comprobaciones", all(checks.values()), str({a: b for a, b in checks.items() if not b} or "todas OK"))
    except Exception as e: add("R2", "telemetry_contracts", False, f"{type(e).__name__}: {e}")
    # R3
    try:
        r = slo_report(RES, T0)
        av = np.mean([x.status in ("ok", "escalated") for x in RES]); p95 = float(np.percentile([x.latency_s for x in RES], 95)); sli = float(np.mean([x.latency_s <= 10 for x in RES]))
        checks = {"disponibilidad": abs(r["availability"] - av) < 1e-9, "p95": abs(r["p95_s"] - p95) < 1e-6, "SLI de latencia": abs(r["latency_sli"] - sli) < 1e-9,
                  "dispara dentro de la caída (10-22)": r["first_fire"] is not None and 10 <= r["first_fire"] <= 22, "sin falsas alarmas": r["false_alarms"] == 0}
        add("R3", "slo_report: 5 comprobaciones", all(checks.values()), f"{ {a: b for a, b in checks.items() if not b} or 'todas OK'} · disponibilidad={r['availability']:.4f} p95={r['p95_s']:.2f} SLI latencia={r['latency_sli']:.3f} primer disparo={r['first_fire']}")
    except Exception as e: add("R3", "slo_report", False, f"{type(e).__name__}: {e}")
    # R4
    try:
        q = quality_report(RES); truth = np.mean([x.grounded for x in RES])
        checks = {"n ≈ 25 %": 0.15 * len(RES) <= q["n"] <= 0.35 * len(RES), "IC contiene la verdad": q["lo"] <= truth <= q["hi"] and abs(q["truth"] - truth) < 1e-9, "cumple el objetivo": q["meets_target"] is True and q["lo"] >= 0.85}
        add("R4", "quality_report: 3 comprobaciones", all(checks.values()), f"{ {a: b for a, b in checks.items() if not b} or 'todas OK'} · p={q['p']:.3f} IC=[{q['lo']:.3f},{q['hi']:.3f}] verdad={truth:.3f}")
    except Exception as e: add("R4", "quality_report", False, f"{type(e).__name__}: {e}")
    # R5
    try:
        s = sampling_report(C, POLICIES); tr = s["error_rate_truth"]
        checks = {"recall de errores = 100 %": s["recall_error"] >= 99.9, "guarda ≤ 35 % de las trazas": s["kept_pct"] <= 35, "tasa corregida a ±15 % de la verdad": abs(s["error_rate_weighted"] - tr) <= 0.15 * tr}
        add("R5", "sampling_report: 3 comprobaciones", all(checks.values()), f"{ {a: b for a, b in checks.items() if not b} or 'todas OK'} · guardadas={s['kept_pct']:.1f}% · verdad={tr:.4f} naive={s['error_rate_naive']:.4f} ponderada={s['error_rate_weighted']:.4f}")
    except Exception as e: add("R5", "sampling_report", False, f"{type(e).__name__}: {e}")
    # R6
    try:
        c = cost_report(SP)
        checks = {"hojas = árbol": abs(c["total_leaf"] - sum(x.cost_usd for x in RES)) < 1e-3 * c["total_leaf"], "doble conteo detectado": c["total_naive"] > c["total_leaf"] * 1.2, "por servicio": set(c["by_service"]) >= {"supportops-orchestrator", "supportops-billing-agent"}}
        add("R6", "cost_report: 3 comprobaciones", all(checks.values()), f"{ {a: b for a, b in checks.items() if not b} or 'todas OK'} · hojas={c['total_leaf']:.4f} ingenua={c['total_naive']:.4f}")
    except Exception as e: add("R6", "cost_report", False, f"{type(e).__name__}: {e}")
    # R7
    try:
        sc = scorecard(MESH, TICKETS, RES); st = checklist_status(CHECKLIST, set(sc[sc.ok]["id"]), set(sc["id"]), EVID); cnt = collections.Counter(x["estado"] for x in st)
        auto_ids = {x["id"] for x in CHECKLIST if x["automatica"]}
        checks = {"31 puntos": len(st) == 31 and all("estado" in x for x in st), "automáticos = scorecard": {x["id"] for x in st if x["estado"] == "medido"} == (set(sc[sc.ok]["id"]) & auto_ids),
                  "≥ 12 evidencia": cnt["evidencia"] >= 12, "3 pendientes": cnt["pendiente"] == 3}
        add("R7", "checklist_status: 4 comprobaciones", all(checks.values()), f"{ {a: b for a, b in checks.items() if not b} or 'todas OK'} · {dict(cnt)}")
    except Exception as e: add("R7", "checklist_status", False, f"{type(e).__name__}: {e}")
    # R8
    try:
        facts = {"slo": slo_report(RES, T0), "quality": quality_report(RES), "sampling": sampling_report(C, POLICIES), "cost": {"total_usd": float(sum(x.cost_usd for x in RES)), "per_ticket": float(np.mean([x.cost_usd for x in RES])), "monthly_full": 1234.5, "monthly_tail": 456.7},
                 "contracts": telemetry_contracts(SP, LG, TICKETS, RES), "checklist": {"medido": 12}}
        doc = architecture_review(facts)
        heads = [l[3:].strip().lower() for l in doc.splitlines() if l.startswith("## ")]
        need = ["contexto", "requisitos", "arquitectura", "decisiones", "riesgos", "coste", "plan", "preguntas"]
        rows_r = [l for l in doc.splitlines() if l.startswith("|") and not set(l) <= set("|- :")]
        checks = {"8 secciones": all(any(n in h for h in heads) for n in need), "≥ 6 riesgos en tabla": len(rows_r) - 1 >= 6, "riesgos con señal y mitigación": "señal" in doc.lower() and "mitigaci" in doc.lower(),
                  "coste con sensibilidad ×2": "×2" in doc and "456" in doc, "plan con la mesa de incidentes": "mesa de incidentes" in doc.lower(), "usa cifras de facts": f"{facts['slo']['p95_s']:.2f}" in doc or f"{facts['quality']['p']:.3f}" in doc}
        add("R8", "architecture_review: 6 comprobaciones", all(checks.values()), str({a: b for a, b in checks.items() if not b} or "todas OK"))
    except Exception as e: add("R8", "architecture_review", False, f"{type(e).__name__}: {e}")
    return pd.DataFrame(rows).set_index("id")

In [ ]:
display(run_rubric())

---
## Retos extra (nivel experto)

1. **Gateway con afinidad**: añade `load_balancing` por `traceID` delante de dos réplicas del gateway (módulo 19) y valida la configuración con `otelcol validate`.
2. **Detector de incidentes de la mesa sobre la plataforma**: ejecuta `incidents.build_incident` y comprueba que **tus contratos** habrían detectado INC-E (trazas partidas) y INC-D (cardinalidad) antes de que un usuario lo notara.
3. **Presupuesto de ingesta**: añade una política `rate_limiting` / `bytes_limiting` al gateway para que un incidente masivo no multiplique la factura (módulo 18) y mide su efecto con el simulador.
4. **Variante de riesgo**: añade un tercer nivel de agentes a tu arquitectura y recalcula amplificación `(r+1)^d`, `num_traces` y el coste de spans.
5. **Defensa oral**: prepara 5 minutos para explicar a un revisor hostil por qué tu plataforma es suficiente… y qué NO cubre.

## Reflexión (producción e hilo conductor)

- De los 31 puntos del checklist, ¿cuáles dependen de **tu** organización y no de tu código? ¿Cómo los conseguirías en un trimestre?
- Si solo pudieras instrumentar tres cosas más, ¿cuáles elegirías y qué incidente de la mesa evitaría cada una?
- ¿Qué parte de este capstone le enseñarías primero a un equipo que empieza (la que da más valor con menos esfuerzo)?

---
# ⛔ SPOILER — intenta resolverlo primero

A continuación, la **solución de referencia** completa y ejecutable (redefine las funciones y vuelve a lanzar la rúbrica).

In [ ]:
# ===================== SOLUCIÓN DE REFERENCIA =====================
def gateway_config(blocked_values, policies):
    cfg = {"extensions": {"health_check": {"endpoint": "0.0.0.0:13133"}},
           "receivers": {"otlp": {"protocols": {"grpc": {"endpoint": "0.0.0.0:4317"}, "http": {"endpoint": "0.0.0.0:4318"}}}},
           "processors": {"memory_limiter": {"check_interval": "1s", "limit_mib": 400, "spike_limit_mib": 100},
                          "redaction": {"allow_all_keys": True, "blocked_values": list(blocked_values), "summary": "info"},
                          "tail_sampling": {"decision_wait": "10s", "num_traces": 50000, "expected_new_traces_per_sec": 100, "policies": policies},
                          "batch": {"timeout": "1s", "send_batch_size": 512}},
           "connectors": {"forward": {}, "span_metrics": {"histogram": {"explicit": {"buckets": ["100ms", "500ms", "1s", "2s", "5s", "10s"]}}, "metrics_flush_interval": "10s"}},
           "exporters": {"otlp_grpc/tempo": {"endpoint": "tempo:4317", "tls": {"insecure": True}}, "otlp_http/loki": {"endpoint": "http://loki:3100/otlp"}, "otlp_http/prometheus": {"endpoint": "http://prometheus:9090/api/v1/otlp"}},
           "service": {"extensions": ["health_check"], "pipelines": {
               "traces/in": {"receivers": ["otlp"], "processors": ["memory_limiter", "redaction"], "exporters": ["forward", "span_metrics"]},
               "traces/sampled": {"receivers": ["forward"], "processors": ["tail_sampling", "batch"], "exporters": ["otlp_grpc/tempo"]},
               "metrics": {"receivers": ["otlp", "span_metrics"], "processors": ["memory_limiter", "batch"], "exporters": ["otlp_http/prometheus"]},
               "logs": {"receivers": ["otlp"], "processors": ["memory_limiter", "redaction", "batch"], "exporters": ["otlp_http/loki"]}}}}
    return yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True, width=200)

def telemetry_contracts(spans, logs, tickets, results):
    by = collections.defaultdict(list)
    for s in spans: by[s["trace_id"]].append(s)
    leaf = sum(int(s["attributes"].get("gen_ai.usage.input_tokens", 0) or 0) for s in spans if s["name"].startswith("chat"))
    roots_ok = True
    for ss in by.values():
        ids = {s["span_id"] for s in ss}; roots = [s for s in ss if not s["parent_span_id"] or s["parent_span_id"] not in ids]
        roots_ok &= len(roots) == 1 and roots[0]["name"].startswith("invoke_workflow")
    txt = [json.dumps(s["attributes"], ensure_ascii=False) for s in spans] + [str(l["body"]) + json.dumps(l["attributes"], ensure_ascii=False) for l in logs]
    pii = sum(v for k, v in P.pii_exposure(txt).items() if k != "name")
    return {"una_traza_por_ticket": len(by) == len(tickets), "hojas_igual_arbol": leaf == sum(r.input_tokens for r in results), "logs_con_trace_id": all(bool(l["trace_id"]) for l in logs),
            "sin_pii_sin_nombres": pii == 0, "workflow_raiz": bool(roots_ok)}

def slo_report(results, t0):
    good = np.zeros(30); tot = np.zeros(30)
    for r in results:
        m = int((r.start_s - t0) // 60)
        if 0 <= m < 30: tot[m] += 1; good[m] += r.latency_s <= 10
    def burn(w):
        cg, ct = np.cumsum(np.insert(good, 0, 0)), np.cumsum(np.insert(tot, 0, 0)); i = np.arange(30); lo = np.maximum(i + 1 - w, 0)
        g, t = cg[i + 1] - cg[lo], ct[i + 1] - ct[lo]
        with np.errstate(divide="ignore", invalid="ignore"): return np.where(t >= 5, (1 - g / np.maximum(t, 1)) / 0.05, 0.0)
    fire = (burn(5) >= 3.0) & (burn(1) >= 3.0)
    return {"availability": float(np.mean([r.status in ("ok", "escalated") for r in results])), "p95_s": float(np.percentile([r.latency_s for r in results], 95)),
            "latency_sli": float(np.mean([r.latency_s <= 10 for r in results])), "first_fire": int(np.argmax(fire)) if fire.any() else None, "false_alarms": int(fire[:10].sum())}

def quality_report(results, every=4):
    uni = [r for r in results if int(hashlib.sha256(r.ticket_id.encode()).hexdigest(), 16) % every == 0]
    k, n = sum(r.grounded for r in uni), len(uni); lo, hi = wilson_interval(k, n); truth = float(np.mean([r.grounded for r in results]))
    return {"n": n, "p": k / n, "lo": lo, "hi": hi, "truth": truth, "covers": lo <= truth <= hi, "meets_target": lo >= 0.85}

def sampling_report(corpus, policies):
    kept = S.TailSampler(policies, decision_wait_s=10, num_traces=50_000).run(corpus.traces).kept
    cov = S.coverage(kept, corpus); w = S.inclusion_weights(kept, lambda t: t.interesting, 0.10)
    return {"recall_error": cov["recall_error_pct"], "kept_pct": cov["kept_traces_pct"], "error_rate_truth": S.estimate_metrics(corpus.traces)["error_rate"],
            "error_rate_naive": S.estimate_metrics(kept)["error_rate"], "error_rate_weighted": S.estimate_metrics(kept, w)["error_rate"]}

def cost_report(spans):
    roll = cost_rollup(spans); naive = 0.0
    for s in spans:
        a = s["attributes"]; i, o = a.get("gen_ai.usage.input_tokens", 0) or 0, a.get("gen_ai.usage.output_tokens", 0) or 0
        if i or o: naive += cost_usd(a.get("gen_ai.response.model") or a.get("gen_ai.request.model") or "mock-small", i, o)
    return {"by_service": roll, "total_leaf": sum(v["cost_usd"] for v in roll.values()), "total_naive": naive}

def checklist_status(items, measured_ok, measured_all, evidence_ids):
    out = []
    for it in items:
        cid = it["id"]
        if it["automatica"]: st = "medido" if cid in measured_ok else ("falla" if cid in measured_all else "sin medir")
        else: st = "evidencia" if cid in evidence_ids else "pendiente"
        out.append({**it, "estado": st})
    return out

RISKS = [("Pérdida silenciosa de telemetría", "`otelcol_exporter_send_failed_*`, `dropped_too_early`, traza canario", "colas con reintento y alertas del pipeline"),
         ("Tormenta de cardinalidad", "series activas ×N con tráfico plano", "View del SDK / aggregate_labels y lint de series"),
         ("Trazas partidas", "trazas por ticket ≠ 1", "contrato n_traces == n_tickets y ParentBased"),
         ("Fuga de PII", "contador de PII en la salida y consulta LogQL de control", "captura apagada, redacción en app y Collector, retención corta"),
         ("Regresión silenciosa", "herramientas por ticket por versión y groundedness", "compuerta apareada, canario SPRT y rollback"),
         ("Cascada de reintentos", "intentos por ticket y timeouts por servicio", "disyuntor, plazos propagados y degradación"),
         ("Coste desbocado", "coste p99 por ticket y trazas con bucles", "LoopGuard, BudgetGuard y max_steps")]

def architecture_review(facts):
    slo, q, smp, cost = facts["slo"], facts["quality"], facts["sampling"], facts["cost"]
    L = ["# Revisión de arquitectura · plataforma de observabilidad de SupportOps", "",
         "## 1. Contexto y objetivos", "SupportOps opera un agente multi-agente (orquestador y 4 especialistas). Objetivo: observar disponibilidad, latencia, calidad, coste y privacidad con un presupuesto acotado.", "",
         "## 2. Requisitos y restricciones", f"Disponibilidad ≥ 99 % (medida {slo['availability']:.4f}), p95 ≤ 8 s (medido {slo['p95_s']:.2f} s), groundedness ≥ 95 % (muestra uniforme: {q['p']:.3f}, IC [{q['lo']:.3f}, {q['hi']:.3f}]), 0 PII en backends.", "",
         "## 3. Arquitectura propuesta", "Agentes → gateway del Collector (memory_limiter, redaction, span_metrics antes del muestreo, tail_sampling, batch) → Tempo, Loki y Prometheus → Grafana y alertas por burn rate.", "",
         "## 4. Decisiones clave", "Tail sampling por valor (errores, lentas, caras, 10 % de base) con RED previo al muestreo; redacción en app y Collector; HMAC para seudónimos.", "",
         "## 5. Riesgos y modos de fallo", "", "| Riesgo | Señal que lo detecta | Mitigación |", "|---|---|---|"]
    L += [f"| {a} | {b} | {c} |" for a, b, c in RISKS]
    L += ["", "## 6. Coste", f"Coste del agente: {cost['per_ticket']:.5f} USD por ticket (ilustrativo). Stack a 1 M tickets/día: {cost['monthly_full']:.1f} USD/mes guardándolo todo frente a {cost['monthly_tail']:.1f} USD/mes con tail sampling. Sensibilidad: ×2 de tráfico duplica trazas y logs; las métricas no cambian.", "",
          "## 7. Plan de despliegue y de pruebas", f"Contratos de telemetría en CI ({sum(facts['contracts'].values())}/{len(facts['contracts'])} en la ejecución de referencia), validación del Collector con otelcol validate y aceptación mediante la mesa de incidentes (los cinco casos en < 15 min con solo telemetría).", "",
          "## 8. Preguntas abiertas", "Despliegue real en un clúster con carga; política de presupuesto de error pactada con producto; límites por tenant."]
    return "\n".join(L)

display(run_rubric())